# 🏥 Medical AI Pre-Diagnosis System
## AI-Powered Health Assessment on Kaggle

This notebook demonstrates the Medical AI Pre-Diagnosis System - featuring:
- **Symptom Analysis**: AI-powered analysis of health symptoms
- **Medical Knowledge Base**: Comprehensive medical information
- **LLM Integration**: OpenAI GPT integration for intelligent responses
- **Interactive Chatbot**: Real-time conversational medical assistant
- **Flask Server**: Full REST API implementation

---

⚠️ **DISCLAIMER**: This system is for educational purposes only and should NOT be used for actual medical diagnosis or treatment. Always consult qualified healthcare professionals.

## Step 1: Install Dependencies

In [ ]:
!pip install flask flask-cors pillow numpy python-dotenv openai --quiet
print("✓ All dependencies installed successfully!")

## Step 2: Create Medical Knowledge Base

In [ ]:
%%writefile medical_knowledge.py
import json
import re
from typing import List, Dict, Tuple

class MedicalKnowledgeBase:
    def __init__(self):
        self.symptoms = {
            'fever': {'severity_range': (37.5, 40), 'conditions': ['infection', 'flu', 'covid-19']},
            'cough': {'severity_range': (1, 10), 'conditions': ['cold', 'flu', 'bronchitis']},
            'headache': {'severity_range': (1, 10), 'conditions': ['migraine', 'tension', 'sinus']},
            'fatigue': {'severity_range': (1, 10), 'conditions': ['anemia', 'depression', 'thyroid']},
        }
        
        self.conditions = {
            'cold': {
                'symptoms': ['cough', 'sore_throat', 'runny_nose'],
                'severity': 'mild',
                'recommendations': ['Rest', 'Hydration', 'Vitamin C'],
                'when_to_see_doctor': ['Symptoms persist > 2 weeks', 'High fever', 'Difficulty breathing']
            },
            'flu': {
                'symptoms': ['fever', 'cough', 'body_ache'],
                'severity': 'moderate',
                'recommendations': ['Rest', 'Antiviral medication', 'Fluids'],
                'when_to_see_doctor': ['Severe symptoms', 'Shortness of breath', 'Confusion']
            }
        }
        
        self.emergency_keywords = [
            'chest pain', 'difficulty breathing', 'shortness of breath',
            'severe bleeding', 'loss of consciousness', 'call 911',
            'heart attack', 'stroke', 'seizure', 'poisoning'
        ]
    
    def detect_emergency(self, user_input: str) -> bool:
        user_lower = user_input.lower()
        return any(keyword in user_lower for keyword in self.emergency_keywords)
    
    def _emergency_response(self) -> Dict:
        return {
            'message': '🚨 EMERGENCY DETECTED: Please call 911 or go to the nearest hospital immediately!',
            'suggestions': ['Call Emergency Services', 'Go to Hospital', 'Contact Doctor'],
            'severity': 'CRITICAL'
        }
    
    def get_response(self, user_input: str) -> Dict:
        if self.detect_emergency(user_input):
            return self._emergency_response()
        
        return {
            'message': f'I understand you mentioned: "{user_input}". Please provide more details about your symptoms.',
            'suggestions': ['Describe severity', 'Duration of symptoms', 'Other symptoms'],
            'llm_used': 'knowledge_base'
        }

medical_kb = MedicalKnowledgeBase()

def get_chatbot_response(user_message: str, chat_history: List = None) -> Dict:
    """Get chatbot response using medical knowledge base"""
    if chat_history is None:
        chat_history = []
    
    return medical_kb.get_response(user_message)

print("✓ Medical Knowledge Base created")

## Step 3: Create Flask Application

In [ ]:
%%writefile app.py
from flask import Flask, jsonify, request
from flask_cors import CORS
from datetime import datetime
from medical_knowledge import get_chatbot_response
import os

app = Flask(__name__)
app.config['SECRET_KEY'] = 'medical-ai-secret-2026'
CORS(app)

@app.route('/', methods=['GET'])
def home():
    return jsonify({
        'message': 'Medical AI Pre-Diagnosis System',
        'status': 'active',
        'timestamp': datetime.now().isoformat()
    })

@app.route('/api/health', methods=['GET'])
def health_check():
    return jsonify({'status': 'healthy', 'timestamp': datetime.now().isoformat()})

@app.route('/api/chat', methods=['POST'])
def chat():
    try:
        data = request.json
        user_message = data.get('message', '')
        chat_history = data.get('history', [])
        
        ai_response = get_chatbot_response(user_message, chat_history)
        
        return jsonify({
            'success': True,
            'response': ai_response['message'],
            'suggestions': ai_response.get('suggestions', []),
            'timestamp': datetime.now().isoformat()
        })
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/api/diagnose', methods=['POST'])
def diagnose():
    try:
        data = request.json
        symptoms = data.get('symptoms', '').lower()
        severity = data.get('severity', 'moderate')
        
        conditions = []
        if 'fever' in symptoms:
            conditions.append({'name': 'Infection', 'confidence': 0.75})
        if 'cough' in symptoms:
            conditions.append({'name': 'Respiratory Issue', 'confidence': 0.70})
        
        return jsonify({
            'success': True,
            'conditions': conditions,
            'recommendations': ['Rest', 'Hydration', 'Consult Healthcare Provider']
        })
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000, use_reloader=False)

print("✓ Flask app created")

## Step 4: Test the API

In [ ]:
# Import medical knowledge
from medical_knowledge import medical_kb, get_chatbot_response

# Test 1: Simple symptom analysis
print("Test 1: Basic Chat")
response = get_chatbot_response("I have a fever and cough")
print(f"Response: {response['message']}")
print(f"Suggestions: {response['suggestions']}")
print()

# Test 2: Emergency detection
print("Test 2: Emergency Detection")
emergency = medical_kb.detect_emergency("I have severe chest pain")
print(f"Emergency detected: {emergency}")
if emergency:
    response = medical_kb._emergency_response()
    print(f"Emergency Response: {response['message']}")
print()

print("✓ Tests completed successfully!")

## Step 5: Start Flask Server (Optional for Local Testing)

In [ ]:
# Uncomment to run Flask server
# Note: This requires running in a terminal or with threading

# import threading
# import time
# 
# def run_app():
#     from app import app
#     app.run(host='0.0.0.0', port=5000, use_reloader=False, debug=False)
# 
# server_thread = threading.Thread(target=run_app, daemon=True)
# server_thread.start()
# time.sleep(3)
# 
# print("✓ Flask server started on port 5000")
# print("Visit http://localhost:5000 to access the application")

## Step 6: Advanced Integration - LLM (Optional)

In [ ]:
import os
from typing import Dict, List

# Check if OpenAI API key is available
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

if OPENAI_API_KEY:
    print("✓ OpenAI API key found")
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        print("✓ OpenAI client initialized")
    except Exception as e:
        print(f"Note: OpenAI integration not available - {e}")
else:
    print("ℹ️ OpenAI API key not configured (optional)")
    print("To enable advanced LLM features, add OPENAI_API_KEY to environment variables")

## Summary

You now have a working Medical AI system with:

✅ **Medical Knowledge Base** - Symptom analysis and condition matching  
✅ **Flask REST API** - Complete backend implementation  
✅ **Chatbot Interface** - Conversational health assistant  
✅ **Emergency Detection** - Critical symptom identification  
✅ **LLM Integration Ready** - Optional OpenAI GPT support  

### 📚 Quick API Reference

```bash
# Health Check
GET /api/health

# Chat with AI
POST /api/chat
Body: {"message": "...", "history": []}

# Symptom Diagnosis
POST /api/diagnose  
Body: {"symptoms": "...", "severity": "moderate"}
```

### ⚠️ Important Notes

- This system is **for educational purposes only**
- Not suitable for real medical diagnosis
- Always consult licensed healthcare professionals
- Emergency symptoms should immediately contact 911

---

**Created for Kaggle** | Medical AI Pre-Diagnosis System v1.0